## Universidad de Buenos Aires
### Aprendizaje Profundo - TP3
Cohorte 25 - 3er bimestre 2026

#### CLASIFICADOR DE EMOCIONES
El objetivo de este trabajo es construir una red neuronal convolucional (CNN) utilizando Pytorch, capaz de clasificar emociones humanas a partir de imágenes faciales. El clasificador deberá identificar una de las 7 emociones básicas: alegría, tristeza, enojo, miedo, sorpresa, disgusto y seriedad. El dataset se encuentra en este link: https://drive.google.com/file/d/1aPHE00zkDhEV1waJKhaOJMdN6-lUc0iT/view?usp=sharing

In [1]:
import gdown
import zipfile
import os

destino = "datos_zip"
DATASET_ROOT_TRAIN = os.path.join(destino, 'dataset_emociones', 'train')
DATASET_ROOT_VAL   = os.path.join(destino, 'dataset_emociones', 'validation')


def _dataset_is_unpacked(*rutes):
    return all(os.path.isdir(rute) and len(os.listdir(rute)) > 0 for rute in rutes)


if _dataset_is_unpacked(DATASET_ROOT_TRAIN, DATASET_ROOT_VAL):
    print('El dataset ya está descargado y descomprimido.')
else:
    url = "https://drive.google.com/uc?id=1aPHE00zkDhEV1waJKhaOJMdN6-lUc0iT"
    output = "archivo.zip"

    gdown.download(url, output, quiet=False)

    os.makedirs(destino, exist_ok=True)
    with zipfile.ZipFile(output, 'r') as zip_ref:
        zip_ref.extractall(destino)

El dataset ya está descargado y descomprimido.


#### 1. Preprocesamiento de Datos (2 puntos)

Antes de entrenar el modelo, se debe analizar qué tipo de preprocesamiento se debe aplicar a las imágenes. Para esto, se puede considerar uno o más aspectos como:

- Tamaño
- Relación de aspecto
- Color o escala de grises
- Cambio de dimensionalidad
- Normalización
- Balanceo de datos
- Data augmentation
- etc.

Sean criteriosos y elijan solo las técnicas que consideren pertinentes para este caso de uso en específico.

Recomendación: usar `torchvision.transforms` para facilitar el preprocesamiento. Lean su documentación si tienen dudas: https://docs.pytorch.org/vision/0.14/transforms.html

Todas las imágenes están en RGB, con un tamaño de 100x100 pixels. Se trata de imágenes de rostros humanos que aparecen centrados (nariz y ojos más o menos al centro de la imagen), por lo que cualquier transformación que apliquemos debería respetar esto, para evitar que el modelo interprete la diferencia como un dato. Sí encontramos un desbalance importante en datos presentes los distintos sets de training: alegria: 4773, seriedad: 2524, tristeza: 1982, sorpresa: 1290, disgusto: 717, enojo: 705, miedo: 281.

Por eso decidimos aplicar las siguientes transformaciones:
- Convertir imágenes a escala de grises: los tres canales de RGB vuelven el procesamiento más pesado y no agregan información al objetivo de reconocer la emoción subyacente.
- Normalización de imágenes: para evitar que mayor o menor brillo o contraste en la imagen sea interpretado por el modelo como un dato relevante.
- Balanceo de datos: para evitar el bias por la diferencia de datos de training entra las clases con menos datos y las clases con más (alegría 4773 vs miedo 281), vamos a usar data augmentation sobre las clases con menos datos y también limitaremos el tamaño de las clases con más datos hasta llegar a números equilibrados.

#### Grayscale y normalization

In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# transformaciones a escala de grises
transform_grayscale = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(root=DATASET_ROOT_TRAIN, transform=transform_grayscale)
val_dataset = datasets.ImageFolder(root=DATASET_ROOT_VAL, transform=transform_grayscale)

# calculamos la media y desviación estándar del conjunto de entrenamiento
loader = DataLoader(train_dataset, batch_size=64, shuffle=False)

mean = 0.0
std = 0.0
total_samples = 0

for images, _ in loader:
    batch_samples = images.size(0)
    images = images.view(batch_samples, images.size(1), -1)
    mean += images.mean(2).sum(0)
    std += images.std(2).sum(0)
    total_samples += batch_samples

mean /= total_samples
std /= total_samples

print(f'Mean: {mean.item()}, Std: {std.item()}')

# transformaciones con normalización
transform_normalized = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean.item()], std=[std.item()])
])

# re-asignamos las transformaciones normalizadas a los datasets de entrenamiento y validación
train_dataset.transform = transform_normalized
val_dataset.transform = transform_normalized

Mean: 0.481630802154541, Std: 0.19285689294338226


#### Data Augmentation y Balancing
Llevamos todas las clases al tamaño definido en `TARGET_SAMPLES_PER_CLASS` para evitar desequilibrios en las sesiones de entrenamientos que puedan generar algún bias en el modelo. Para esto aplicamos data augmentation a las clases que no llegan a la cantidad de muestras en el target y restringimos las muestras en las clases que lo superan.

In [3]:
# aplicamos data-augmentation sobre las clases con menos muestras
# y limitamos las clases con más muestras, para balancear el dataset de entrenamiento

import random
from collections import defaultdict
from PIL import Image

# tamaño muestras por clase, elegido arbitrariamente (pero aprox el promedio de todas las clases)
# modificar si se desea un balance diferente
TARGET_SAMPLES_PER_CLASS = 1500

# transformaciones aplicadas: horizontal flip, rotaciones leves y jitter de brillo/contraste
transform_augmented = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean.item()], std=[std.item()]),
])


class BalancedEmotionDataset(DataLoader):
    """Sub-muestrea las clases con más imágenes que target_per_class y aplica
    data augmentation a las clases con menos, hasta emparejar todas las clases
    a target_per_class muestras"""

    def __init__(self, image_folder_dataset, target_per_class, base_transform, augment_transform, seed=42):
        rng = random.Random(seed)

        samples_by_class = defaultdict(list)
        for path, label in image_folder_dataset.samples:
            samples_by_class[label].append(path)

        # cada item es (path, label, aplicar_augmentation)
        self.items = []
        for label, paths in samples_by_class.items():
            if len(paths) >= target_per_class:
                selected = rng.sample(paths, target_per_class)
                self.items.extend((p, label, False) for p in selected)
            else:
                self.items.extend((p, label, False) for p in paths)
                faltantes = target_per_class - len(paths)
                extra_paths = [rng.choice(paths) for _ in range(faltantes)]
                self.items.extend((p, label, True) for p in extra_paths)

        self.base_transform = base_transform
        self.augment_transform = augment_transform
        self.classes = image_folder_dataset.classes

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label, augment = self.items[idx]
        image = Image.open(path).convert('RGB')
        transform = self.augment_transform if augment else self.base_transform
        return transform(image), label

train_dataset_balanced = BalancedEmotionDataset(
    train_dataset,
    target_per_class=TARGET_SAMPLES_PER_CLASS,
    base_transform=transform_normalized,
    augment_transform=transform_augmented,
)

# verificamos el balance final
counts = defaultdict(int)
for _, label, _ in train_dataset_balanced.items:
    counts[label] += 1
for idx, class_name in enumerate(train_dataset_balanced.classes):
    print(f'{class_name}: {counts[idx]}')

# Nota: de acá en más usamos train_dataset_balanced en lugar de train_dataset

alegria: 1500
disgusto: 1500
enojo: 1500
miedo: 1500
seriedad: 1500
sorpresa: 1500
tristeza: 1500
